# 🥈 TASK 3 — Silver Layer Transformation
**Notebook:** `silver_layer_transformation`  
**Depends on:** `healthcare_pipeline` (Tasks 1 & 2 must be run first)  
**Steps covered:**
- Step 12 — Cast date columns, compute patient_age_at_visit
- Step 13 — Compute total_bill and patient_liability
- Step 14 — Add visit_category (long_stay / short_stay / day_visit)
- Step 15 — Three-way join: visits + patients + billing
- Step 16 — Write silver.patient_visits partitioned by department
- Step 17 — SCD Type 2 MERGE for silver.patients
- Step 18 — Enable Change Data Feed & query the change log
- Step 19 — OPTIMIZE + ZORDER BY (patient_id, visit_date)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Imports & Spark configuration
# ─────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

spark = SparkSession.builder.getOrCreate()

# Enable Change Data Feed globally (also set per-table below)
spark.conf.set("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")
# Allow schema merges during writes (SCD Type 2 merge)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
# Keep partitions low for dev clusters
spark.conf.set("spark.sql.shuffle.partitions", "8")

print(f"✅ Spark {spark.version} ready")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Catalog / schema / table references
# Match EXACTLY the names written in healthcare_pipeline notebook
# ─────────────────────────────────────────────────────────────
CATALOG = "databricks_project_tsv-752"   # ← your Unity Catalog name
SCHEMA  = "healthcare"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

# ── Bronze source tables (written in Task 2) ──────────────────
BRONZE_VISITS   = f"`{CATALOG}`.`{SCHEMA}`.bronze_patient_visits"
BRONZE_PATIENTS = f"`{CATALOG}`.`{SCHEMA}`.bronze_patients"
BRONZE_BILLING  = f"`{CATALOG}`.`{SCHEMA}`.bronze_billing"

# ── Silver target tables ──────────────────────────────────────
SILVER_VISITS   = f"`{CATALOG}`.`{SCHEMA}`.silver_patient_visits"
SILVER_PATIENTS = f"`{CATALOG}`.`{SCHEMA}`.silver_patients"

# ── Volume paths (same root used in Task 1) ───────────────────
BASE_VOL        = f"/Volumes/{CATALOG}/{SCHEMA}"
SILVER_PATH     = f"{BASE_VOL}/silver/"
CHECKPOINT_PATH = f"{BASE_VOL}/checkpoints/silver_cdf/"

print(f"✅ Context: {CATALOG}.{SCHEMA}")
print(f"   Bronze visits   → {BRONZE_VISITS}")
print(f"   Bronze patients → {BRONZE_PATIENTS}")
print(f"   Bronze billing  → {BRONZE_BILLING}")
print(f"   Silver visits   → {SILVER_VISITS}")
print(f"   Silver patients → {SILVER_PATIENTS}")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Read the three Bronze tables
# Validate row counts before any transformation
# ─────────────────────────────────────────────────────────────
df_visits_bronze   = spark.table(BRONZE_VISITS)
df_patients_bronze = spark.table(BRONZE_PATIENTS)
df_billing_bronze  = spark.table(BRONZE_BILLING)

print("📊 Bronze row counts:")
print(f"   patient_visits : {df_visits_bronze.count():>6,}")
print(f"   patients       : {df_patients_bronze.count():>6,}")
print(f"   billing        : {df_billing_bronze.count():>6,}")

print("\n📋 Visits schema (incoming):")
df_visits_bronze.printSchema()

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — STEP 12
# Cast visit_date → DateType  (already Date from inferSchema;
#   explicit cast ensures correctness after any schema drift)
# Cast date_of_birth → DateType
# Compute patient_age_at_visit = year(visit_date) - year(dob)
#   using months_between for an accurate year-fraction approach
# ─────────────────────────────────────────────────────────────

# -- Transform visits: explicit DateType cast + age derivation --
df_visits_cast = (
    df_visits_bronze
    .withColumn("visit_date",
                F.col("visit_date").cast(DateType()))
)

# -- Transform patients: explicit DateType cast --
df_patients_cast = (
    df_patients_bronze
    .withColumn("date_of_birth",
                F.col("date_of_birth").cast(DateType()))
)

# We will compute patient_age_at_visit AFTER joining visits with patients
# (so we have both visit_date and date_of_birth in one DataFrame)

print("✅ Date columns explicitly cast to DateType")
df_visits_cast.select("visit_id", "visit_date").show(5, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — STEP 13
# Financial metrics on the billing DataFrame:
#   total_bill      = consultation_fee + medication_cost + procedure_cost
#   patient_liability = total_bill - insurance_covered
# Cast all monetary columns to DoubleType for consistent precision
# ─────────────────────────────────────────────────────────────

df_billing_enriched = (
    df_billing_bronze
    # Ensure all monetary columns are DoubleType
    .withColumn("consultation_fee",  F.col("consultation_fee").cast(DoubleType()))
    .withColumn("medication_cost",   F.col("medication_cost").cast(DoubleType()))
    .withColumn("procedure_cost",    F.col("procedure_cost").cast(DoubleType()))
    .withColumn("insurance_covered", F.col("insurance_covered").cast(DoubleType()))
    # Derived financial columns
    .withColumn(
        "total_bill",
        F.round(
            F.col("consultation_fee") +
            F.col("medication_cost")  +
            F.col("procedure_cost"),
            2
        )
    )
    .withColumn(
        "patient_liability",
        F.round(
            F.col("total_bill") - F.col("insurance_covered"),
            2
        )
    )
)

print("✅ Financial metrics computed")
display(
    df_billing_enriched
    .select("bill_id", "visit_id", "total_bill", "insurance_covered", "patient_liability")
    .limit(5)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — STEP 14
# visit_category classification based on duration_days:
#   duration_days >  5  → long_stay
#   duration_days >= 2  → short_stay   (i.e. 2–5 inclusive)
#   duration_days == 1  → day_visit
#   else               → unknown (safety net for nulls / 0)
# ─────────────────────────────────────────────────────────────

df_visits_categorized = (
    df_visits_cast
    .withColumn(
        "visit_category",
        F.when(F.col("duration_days") > 5,  F.lit("long_stay"))
         .when(F.col("duration_days") >= 2, F.lit("short_stay"))
         .when(F.col("duration_days") == 1, F.lit("day_visit"))
         .otherwise(F.lit("unknown"))
    )
)

print("✅ visit_category column added")
print("\n📊 Distribution of visit_category:")
display(
    df_visits_categorized
    .groupBy("visit_category")
    .count()
    .orderBy("visit_category")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — STEP 15
# Three-way join:
#   visits (left) + patients (left join on patient_id)
#                 + billing  (left join on visit_id)
# Compute patient_age_at_visit after join (needs both date cols)
# Drop Bronze metadata cols from patients & billing to keep schema clean
# ─────────────────────────────────────────────────────────────

# Select only the demographic columns we need from patients
# (avoid column name collisions from _ingested_at / _source_file)
df_patients_slim = df_patients_cast.select(
    "patient_id",
    "full_name",
    "date_of_birth",
    "gender",
    "blood_group",
    "contact_number",
    "city"
)

# Select billing columns needed (drop bronze metadata)
df_billing_slim = df_billing_enriched.select(
    "visit_id",
    "bill_id",
    "consultation_fee",
    "medication_cost",
    "procedure_cost",
    "insurance_covered",
    "payment_status",
    "total_bill",
    "patient_liability"
)

# Join 1: visits ← patients  (left join; some visits may not match if data gaps)
df_joined = df_visits_categorized.join(
    df_patients_slim,
    on="patient_id",
    how="left"
)

# Join 2: result ← billing  (left join; same visit_id key)
df_joined = df_joined.join(
    df_billing_slim,
    on="visit_id",
    how="left"
)

# Compute patient_age_at_visit now that both date columns are present
# floor(months_between / 12) gives exact full years
df_silver_visits = (
    df_joined
    .withColumn(
        "patient_age_at_visit",
        F.floor(
            F.months_between(F.col("visit_date"), F.col("date_of_birth")) / 12
        ).cast("integer")
    )
    # Add Silver-layer audit timestamp
    .withColumn("_silver_processed_at", F.current_timestamp())
)

print(f"✅ Three-way join complete — {df_silver_visits.count():,} rows")
print("\n📋 Final Silver visits schema:")
df_silver_visits.printSchema()
display(df_silver_visits.select(
    "visit_id", "patient_id", "visit_date",
    "patient_age_at_visit", "total_bill",
    "patient_liability", "visit_category"
).limit(5))

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — STEP 16
# Write enriched data to silver.patient_visits
# Partitioned by department for optimised reads per department
# Enable Change Data Feed on the table via table property
# Idempotent: overwrite mode + overwriteSchema=true for re-runs
# ─────────────────────────────────────────────────────────────

(
    df_silver_visits
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    # Partition by department — each department gets its own file set
    .partitionBy("department")
    .option("delta.enableChangeDataFeed", "true")   # CDF at table creation
    .saveAsTable(SILVER_VISITS)
)

# Also enable CDF via ALTER TABLE for tables that already existed
spark.sql(f"""
    ALTER TABLE {SILVER_VISITS}
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

cnt = spark.table(SILVER_VISITS).count()
print(f"✅ {SILVER_VISITS} written — {cnt:,} rows, partitioned by department")

# Verify partitions written
print("\n📂 Partition distribution (by department):")
display(
    spark.table(SILVER_VISITS)
    .groupBy("department")
    .count()
    .orderBy("department")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — STEP 17  (Part A)
# SCD Type 2 — Create silver.patients table with SCD columns
# SCD columns:
#   is_current           BOOLEAN  — True = active record
#   effective_start_date DATE     — when this version became active
#   effective_end_date   DATE     — when this version was superseded (NULL if current)
#
# Initial load: all records are current, start = today, end = NULL
# ─────────────────────────────────────────────────────────────

TODAY = F.current_date()
HIGH_DATE = F.lit("9999-12-31").cast(DateType())  # sentinel for active rows

# Build the initial silver patients DataFrame with SCD columns
df_patients_scd_initial = (
    df_patients_cast
    .select(
        "patient_id", "full_name", "date_of_birth",
        "gender", "blood_group", "contact_number", "city"
    )
    .withColumn("is_current",           F.lit(True))
    .withColumn("effective_start_date", TODAY.cast(DateType()))
    .withColumn("effective_end_date",   HIGH_DATE)
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# Create (or overwrite) silver.patients with SCD structure
(
    df_patients_scd_initial
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(SILVER_PATIENTS)
)

print(f"✅ {SILVER_PATIENTS} created with SCD Type 2 columns — {spark.table(SILVER_PATIENTS).count():,} rows")
display(spark.table(SILVER_PATIENTS).limit(5))

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — STEP 17  (Part B)
# SCD Type 2 — MERGE INTO to process the patients_update batch
# (patients_update.csv was uploaded in Task 2 alongside the other CSVs)
#
# Logic:
#   WHEN MATCHED AND city has changed:
#     → expire old record (is_current=False, effective_end_date=today-1)
#     → insert new record  (is_current=True, effective_start_date=today)
#   WHEN NOT MATCHED:
#     → insert brand-new patient
#
# We implement the two-step approach because Delta MERGE cannot
# simultaneously expire AND insert within a single MATCHED clause.
# Step 1 → expire changed rows via MERGE UPDATE
# Step 2 → insert new versions via INSERT
# ─────────────────────────────────────────────────────────────

BASE_VOL = f"/Volumes/{CATALOG}/{SCHEMA}"
PATIENTS_UPDATE_CSV = f"{BASE_VOL}/raw/patients_update.csv"

# Read the incremental update file
df_patients_update = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATIENTS_UPDATE_CSV)
    .withColumn("date_of_birth", F.col("date_of_birth").cast(DateType()))
)

print(f"📥 patients_update records: {df_patients_update.count()}")
display(df_patients_update)

# Register update data as a temp view for SQL MERGE
df_patients_update.createOrReplaceTempView("patients_updates_staging")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 11 — STEP 17  (Part C)
# SCD Type 2 — STEP 1: Expire changed rows
# Update existing current records where city has changed:
#   set is_current = False
#   set effective_end_date = today - 1 day  (day before new version)
# ─────────────────────────────────────────────────────────────

delta_patients = DeltaTable.forName(spark, SILVER_PATIENTS)

delta_patients.alias("target").merge(
    df_patients_update.alias("source"),
    """
        target.patient_id = source.patient_id
        AND target.is_current = true
        AND target.city <> source.city
    """
).whenMatchedUpdate(
    set={
        "is_current"          : F.lit(False),
        "effective_end_date"  : F.date_sub(F.current_date(), 1),
    }
).execute()

print("✅ Step 1 complete — expired old records where city changed")

# Show rows that are now expired
print("\n📋 Expired rows (is_current = False):")
display(
    spark.table(SILVER_PATIENTS)
    .filter(F.col("is_current") == False)
    .select("patient_id", "city", "is_current",
            "effective_start_date", "effective_end_date")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 12 — STEP 17  (Part D)
# SCD Type 2 — STEP 2: Insert new versions for changed patients
# AND insert completely new patients (NOT MATCHED)
# ─────────────────────────────────────────────────────────────

# Identify patients whose old record was just expired (these need new rows)
# We join update file with current silver to find which ones were expired
df_expired_patient_ids = (
    spark.table(SILVER_PATIENTS)
    .filter(F.col("is_current") == False)
    .filter(F.col("effective_end_date") == F.date_sub(F.current_date(), 1))
    .select("patient_id")
)

# New versions to insert — only for patients whose record was expired
df_new_versions = (
    df_patients_update
    .join(df_expired_patient_ids, on="patient_id", how="inner")
    .withColumn("is_current",           F.lit(True))
    .withColumn("effective_start_date", F.current_date().cast(DateType()))
    .withColumn("effective_end_date",   F.lit("9999-12-31").cast(DateType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# Brand-new patients (not in silver table at all)
df_existing_ids = spark.table(SILVER_PATIENTS).select("patient_id").distinct()
df_brand_new = (
    df_patients_update
    .join(df_existing_ids, on="patient_id", how="left_anti")
    .withColumn("is_current",           F.lit(True))
    .withColumn("effective_start_date", F.current_date().cast(DateType()))
    .withColumn("effective_end_date",   F.lit("9999-12-31").cast(DateType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# Combine new versions + brand new patients
df_to_insert = df_new_versions.unionByName(df_brand_new)

# Append new version rows
(
    df_to_insert
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(SILVER_PATIENTS)
)

print(f"✅ Step 2 complete — new SCD rows inserted: {df_to_insert.count()}")

print("\n📋 Full silver.patients after SCD Type 2:")
display(
    spark.table(SILVER_PATIENTS)
    .select("patient_id", "full_name", "city",
            "is_current", "effective_start_date", "effective_end_date")
    .orderBy("patient_id", "effective_start_date")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 13 — STEP 18  (Part A)
# Confirm Change Data Feed is enabled on silver.patient_visits
# Query the CDF change log from version 0 onwards
# _change_type values: insert, update_preimage, update_postimage, delete
# ─────────────────────────────────────────────────────────────

# Verify CDF property
print("📋 Table properties for silver.patient_visits:")
spark.sql(f"SHOW TBLPROPERTIES {SILVER_VISITS}").show(truncate=False)

# Read the change data feed from version 0 (captures all changes)
df_cdf = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table(SILVER_VISITS)
)

print(f"\n📊 CDF total change records: {df_cdf.count():,}")
print("\n📋 CDF sample — visit_id, _change_type, _commit_version, _commit_timestamp:")
display(
    df_cdf
    .select("visit_id", "department", "_change_type",
            "_commit_version", "_commit_timestamp")
    .orderBy("_commit_version", "visit_id")
    .limit(20)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 14 — STEP 18  (Part B)
# Simulate an incremental update to demonstrate CDF capturing changes
# Update discharge_status for a subset of rows, then re-read CDF
# ─────────────────────────────────────────────────────────────

# Simulate an update: re-classify a few discharge statuses
spark.sql(f"""
    UPDATE {SILVER_VISITS}
    SET discharge_status = 'reviewed'
    WHERE discharge_status = 'recovered'
    AND department = 'Cardiology'
""")

print("✅ Simulated UPDATE on Cardiology recovered → reviewed")

# Re-read CDF — now includes update_preimage and update_postimage rows
df_cdf_v2 = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table(SILVER_VISITS)
)

print(f"\n📊 CDF records after UPDATE: {df_cdf_v2.count():,}")
print("\n📋 Change types breakdown:")
display(
    df_cdf_v2
    .groupBy("_change_type", "_commit_version")
    .count()
    .orderBy("_commit_version")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 15 — STEP 19
# OPTIMIZE silver.patient_visits with ZORDER BY (patient_id, visit_date)
#
# OPTIMIZE  → compacts small files into larger Parquet files
#              (reduces file count, improves scan throughput)
# ZORDER BY → co-locates related data within each Parquet file
#              so that queries filtering on patient_id + visit_date
#              can skip large portions of data (data skipping)
# ─────────────────────────────────────────────────────────────

print("⏳ Running OPTIMIZE + ZORDER on silver.patient_visits ...")
print("   (This may take a few minutes on large clusters)")

optimize_result = spark.sql(f"""
    OPTIMIZE {SILVER_VISITS}
    ZORDER BY (patient_id, visit_date)
""")

print("✅ OPTIMIZE complete")
display(optimize_result)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 16 — DESCRIBE HISTORY & Final Validation
# Confirm all operations were logged in the Delta transaction log
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print(f"📜 DESCRIBE HISTORY: {SILVER_VISITS}")
print("=" * 70)
display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_VISITS}")
    .select("version", "timestamp", "operation", "operationMetrics")
)

print("\n" + "=" * 70)
print("🏆 SILVER LAYER — TRANSFORMATION SUMMARY")
print("=" * 70)
silver_tables = {
    "silver_patient_visits" : SILVER_VISITS,
    "silver_patients"       : SILVER_PATIENTS,
}
for label, tbl in silver_tables.items():
    cnt = spark.table(tbl).count()
    print(f"  ✅ {label:30s}  {cnt:>6,} rows  →  {tbl}")

print("\n📊 Sample enriched Silver visit record:")
display(
    spark.table(SILVER_VISITS)
    .select(
        "visit_id", "patient_id", "visit_date",
        "department", "visit_category",
        "patient_age_at_visit",
        "total_bill", "patient_liability",
        "discharge_status"
    )
    .limit(10)
)